# Combine Agent Splits

In [1]:
# Combine Agent Language Splits
# This notebook combines language-split files (e.g., copilot-*, cursor-*, jules-*)
# into single files per agent and removes the split files

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.utils.config import AI_PR_DIR, HUMAN_REVIEW_DIR

print(f"Working directory: {Path.cwd().relative_to(PROJECT_ROOT) if Path.cwd().is_relative_to(PROJECT_ROOT) else Path.cwd()}")
print(f"Project root: {PROJECT_ROOT.name}")
print(f"AI PR directory: {AI_PR_DIR.relative_to(PROJECT_ROOT)}")
print(f"Human review directory: {HUMAN_REVIEW_DIR.relative_to(PROJECT_ROOT)}\n")

# Agents with language splits
AGENTS_TO_COMBINE = ["copilot", "cursor", "jules", "codex"]


def combine_files(directory: Path, prefix: str, agent: str, dry_run: bool = True):
    """
    Combine all files matching pattern {prefix}_{agent}-*.jsonl into {prefix}_{agent}.jsonl

    Args:
        directory: Directory containing the files
        prefix: File prefix (e.g., "ai_authored" or "human_reviews")
        agent: Agent name (e.g., "copilot")
        dry_run: If True, only show what would be done without actually doing it
    """
    # Find all split files
    pattern = f"{prefix}_{agent}-*.jsonl"
    split_files = sorted(directory.glob(pattern))

    if not split_files:
        print(f"⚠️ No files found matching {pattern}")
        return None

    # Output file
    combined_file = directory / f"{prefix}_{agent}.jsonl"

    print(f"\n{'='*80}")
    print(f"Agent: {agent.upper()}")
    print(f"{'='*80}")
    print(f"Found {len(split_files)} split files:")

    # Count lines in each file
    line_counts = {}
    total_lines = 0
    for f in split_files:
        with open(f, "r") as file:
            count = sum(1 for _ in file)
            line_counts[f.name] = count
            total_lines += count
        print(f"  - {f.name}: {count:,} lines")

    print(f"\nTotal lines to combine: {total_lines:,}")
    print(f"Output file: {combined_file.name}")

    if dry_run:
        print("  [DRY RUN - no files modified]")
        return {
            "agent": agent,
            "split_files": split_files,
            "combined_file": combined_file,
            "total_lines": total_lines,
            "line_counts": line_counts,
        }

    # Actually combine files
    print("\nCombining files...")
    lines_written = 0
    with open(combined_file, "w") as outfile:
        for split_file in split_files:
            with open(split_file, "r") as infile:
                for line in infile:
                    outfile.write(line)
                    lines_written += 1

    print(f"✅ Wrote {lines_written:,} lines to {combined_file.name}")

    # Verify
    with open(combined_file, "r") as f:
        actual_lines = sum(1 for _ in f)

    if actual_lines == total_lines:
        print(f"✅ Verification passed: {actual_lines:,} lines")
    else:
        print(f"❌ Verification FAILED: Expected {total_lines:,}, got {actual_lines:,}")
        return None

    return {
        "agent": agent,
        "split_files": split_files,
        "combined_file": combined_file,
        "total_lines": total_lines,
        "verified": actual_lines == total_lines,
    }

Working directory: notebooks
Project root: aidev-extended
AI PR directory: data\raw\ai_authored_prs
Human review directory: data\raw\human_reviews



In [2]:
# ============================================================================
# STEP 1: DRY RUN - Show what will be done
# ============================================================================
print("\n" + "=" * 80)
print("STEP 1: DRY RUN - Previewing changes")
print("=" * 80)

dry_run_results = {}

for agent in AGENTS_TO_COMBINE:
    # AI PRs
    result = combine_files(AI_PR_DIR, "ai_authored", agent, dry_run=True)
    if result:
        dry_run_results[f"ai_authored_{agent}"] = result

    # Human Reviews
    result = combine_files(HUMAN_REVIEW_DIR, "human_reviews", agent, dry_run=True)
    if result:
        dry_run_results[f"human_reviews_{agent}"] = result


STEP 1: DRY RUN - Previewing changes

Agent: COPILOT
Found 15 split files:
  - ai_authored_copilot-c.jsonl: 522 lines
  - ai_authored_copilot-cpp.jsonl: 2,091 lines
  - ai_authored_copilot-csharp.jsonl: 5,873 lines
  - ai_authored_copilot-go.jsonl: 8,603 lines
  - ai_authored_copilot-html.jsonl: 257 lines
  - ai_authored_copilot-java.jsonl: 1,490 lines
  - ai_authored_copilot-javascript.jsonl: 1,145 lines
  - ai_authored_copilot-kotlin.jsonl: 506 lines
  - ai_authored_copilot-other.jsonl: 2,569 lines
  - ai_authored_copilot-php.jsonl: 746 lines
  - ai_authored_copilot-powershell.jsonl: 111 lines
  - ai_authored_copilot-python.jsonl: 3,292 lines
  - ai_authored_copilot-ruby.jsonl: 379 lines
  - ai_authored_copilot-rust.jsonl: 1,330 lines
  - ai_authored_copilot-typescript.jsonl: 6,861 lines

Total lines to combine: 35,775
Output file: ai_authored_copilot.jsonl
  [DRY RUN - no files modified]

Agent: COPILOT
Found 15 split files:
  - human_reviews_copilot-c.jsonl: 422 lines
  - human_re

In [3]:
# ============================================================================
# STEP 2: User Confirmation
# ============================================================================
print("\n" + "=" * 80)
print("STEP 2: Review and Confirm")
print("=" * 80)

total_splits_to_delete = sum(len(r["split_files"]) for r in dry_run_results.values())
print(f"\nThis will:")
print(f"  Create {len(dry_run_results)} combined files")
print(f"  Delete {total_splits_to_delete} split files after verification")

# Change to "YES" below to actually execute
CONFIRM = "YES"  # Change to "YES" to execute


STEP 2: Review and Confirm

This will:
  Create 8 combined files
  Delete 68 split files after verification


In [ ]:
# ============================================================================
# STEP 3: Execute (if confirmed)
# ============================================================================
if CONFIRM == "YES":
    print("\n" + "=" * 80)
    print("STEP 3: EXECUTING - Combining and deleting files")
    print("=" * 80)

    execution_results = {}

    for agent in AGENTS_TO_COMBINE:
        # AI PRs
        result = combine_files(AI_PR_DIR, "ai_authored", agent, dry_run=False)
        if result and result["verified"]:
            execution_results[f"ai_authored_{agent}"] = result

        # Human Reviews
        result = combine_files(HUMAN_REVIEW_DIR, "human_reviews", agent, dry_run=False)
        if result and result["verified"]:
            execution_results[f"human_reviews_{agent}"] = result

    # ============================================================================
    # STEP 4: Delete split files (only if all verifications passed)
    # ============================================================================
    print("\n" + "=" * 80)
    print("STEP 4: Deleting original split files")
    print("=" * 80)

    all_verified = all(r["verified"] for r in execution_results.values())

    if all_verified:
        print("✅ All combinations verified successfully")
        print("\n🗑️ Deleting split files...\n")

        deleted_count = 0
        for key, result in execution_results.items():
            for split_file in result["split_files"]:
                print(f"  Deleting: {split_file.name}")
                split_file.unlink()
                deleted_count += 1

        print(f"\n✅ Deleted {deleted_count} split files")

    else:
        print("❌ Some verifications failed - NOT deleting any files")
        print("    Please check the results above")

    # ============================================================================
    # STEP 5: Final Summary
    # ============================================================================
    print("\n" + "=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)

    for key, result in execution_results.items():
        print(f"\n{key}:")
        print(f"  Combined: {result['combined_file'].name}")
        print(f"  Lines: {result['total_lines']:,}")
        print(f"  Verified: {'✅' if result['verified'] else '❌'}")
        print(f"  Split files deleted: {len(result['split_files'])}")

else:
    print("\n⚠️ Execution skipped (CONFIRM != 'YES')")
    print("    To execute, change CONFIRM = 'YES' in the cell above and re-run")
    print("\nDry run results:")
    for key, result in dry_run_results.items():
        print(f"  - {key}: {result['total_lines']:,} lines from {len(result['split_files'])} files")


STEP 3: EXECUTING - Combining and deleting files

Agent: COPILOT
Found 15 split files:
  - ai_authored_copilot-c.jsonl: 522 lines
  - ai_authored_copilot-cpp.jsonl: 2,091 lines
  - ai_authored_copilot-csharp.jsonl: 5,873 lines
  - ai_authored_copilot-go.jsonl: 8,603 lines
  - ai_authored_copilot-html.jsonl: 257 lines
  - ai_authored_copilot-java.jsonl: 1,490 lines
  - ai_authored_copilot-javascript.jsonl: 1,145 lines
  - ai_authored_copilot-kotlin.jsonl: 506 lines
  - ai_authored_copilot-other.jsonl: 2,569 lines
  - ai_authored_copilot-php.jsonl: 746 lines
  - ai_authored_copilot-powershell.jsonl: 111 lines
  - ai_authored_copilot-python.jsonl: 3,292 lines
  - ai_authored_copilot-ruby.jsonl: 379 lines
  - ai_authored_copilot-rust.jsonl: 1,330 lines
  - ai_authored_copilot-typescript.jsonl: 6,861 lines

Total lines to combine: 35,775
Output file: ai_authored_copilot.jsonl

Combining files...
✅ Wrote 35,775 lines to ai_authored_copilot.jsonl
✅ Verification passed: 35,775 lines

Agent: C